# Ground Truth Dataset Generation

This notebook generates a synthetic evaluation dataset for the Yoga RAG system.

## Approach

We'll use an LLM to generate diverse, realistic questions based on the yoga pose data, along with:
- The question text
- The expected answer
- The relevant pose ID(s) for verification

## Question Types

1. **Identify pose** - "What pose helps with lower back pain?"
2. **Benefits** - "What are the benefits of Downward-Facing Dog?"
3. **Category** - "Which poses are good for balance?"
4. **Difficulty** - "What are some beginner-friendly standing poses?"
5. **Instructions** - "How do I perform Tree Pose?"
6. **Contraindications** - "Which poses should I avoid with knee injuries?"
7. **Modifications** - "How can I modify Pigeon Pose if I have tight hips?"
8. **Confusion pairs** - "What's the difference between Warrior I and Warrior II?"

Target: 50-80 diverse questions

In [1]:
import pandas as pd
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables
load_dotenv()

# Load the merged yoga dataset
df = pd.read_csv('../data/yoga_data_merged.csv')
print(f"Loaded {len(df)} yoga poses")
df.head()

Loaded 202 yoga poses


,id,pose_name,sanskrit_name,category,difficulty_level,benefits,contraindications,breathing_pattern,duration_or_reps,modifications,instructions
0,1,Mountain Pose,Tadasana,standing,beginner,"Mountain Pose strengthens the ankles, calves, ...",Avoid practicing Mountain Pose if you have sev...,Inhale: Feel the chest expand and the spine le...,"Hold Mountain Pose for 3-5 breaths, or as long...","For beginners, use a block or a wall for suppo...","1. Stand with your feet hip-width apart, paral..."
1,2,Tree Pose,Vrksasana,balancing,beginner,"Tree Pose (Vrksasana) improves balance, stabil...",Avoid Tree Pose if you have ankle or knee inju...,"Inhale deeply and lengthen your spine, exhale ...",Hold Tree Pose for 30 seconds to 1 minute on e...,Use a block or wall for support if you're new ...,"1. Start by standing on one leg, with the othe..."
2,3,Downward-Facing Dog,Adho Mukha Svanasana,standing,beginner,Downward-Facing Dog stretches and strengthens ...,Avoid this pose if you have recent injuries to...,"Inhale as you lift your hips up and back, exha...","Hold for 3-5 breaths, or for 30-60 seconds. Re...","For beginners or those with wrist issues, try ...","1. Start on all fours, with your hands shoulde..."
3,4,Cobra Pose,Bhujangasana,backbend,beginner,"Cobra Pose strengthens the back muscles, opens...",Avoid Cobra Pose if you have a recent back inj...,Inhale as you press your palms into the ground...,"Hold Cobra Pose for 3-5 breaths, or 15-30 seco...","For a gentler variation, place your forearms o...",1. Lie on your stomach with your hands under y...
4,5,Cat-Cow Pose,Marjaryasana-Bitilasana,standing,beginner,"The Cat-Cow Pose stretches the spine, neck, an...","This pose is generally safe for most people, b...",Inhale as you arch your back and lift your hea...,"Repeat the sequence for 5-10 breaths, moving s...","For a more gentle version, try using a block o...","1. Start on your hands and knees, with your wr..."


## Setup LLM API Connection

Using Hyperbolic or Nebius API (OpenAI-compatible)

In [2]:
# Configure LLM API
llm_provider = os.getenv('LLM_PROVIDER', 'hyperbolic')
api_key = os.getenv('LLM_API_KEY')
base_url = os.getenv('LLM_BASE_URL', 'https://api.hyperbolic.xyz/v1')
model = os.getenv('LLM_MODEL', 'meta-llama/Llama-3.3-70B-Instruct')

client = OpenAI(
    api_key=api_key,
    base_url=base_url
)

print(f"Using {llm_provider} with model: {model}")

Using hyperbolic with model: meta-llama/Meta-Llama-3.1-70B-Instruct


## Define Question Templates

We'll create templates for different question types to ensure diversity.

In [3]:
question_templates = {
    'identify_pose': [
        "What yoga pose helps with {benefit}?",
        "Which pose is good for {benefit}?",
        "I want to improve {benefit}. What pose should I do?",
        "What's a good pose for {benefit}?"
    ],
    'benefits': [
        "What are the benefits of {pose_name}?",
        "Why should I practice {pose_name}?",
        "What does {pose_name} do for my body?",
        "How does {pose_name} help me?"
    ],
    'category': [
        "What are some {category} poses?",
        "Which poses are {category} poses?",
        "Can you recommend {category} poses?",
        "I want to practice {category} poses. What should I do?"
    ],
    'difficulty': [
        "What are some {difficulty} {category} poses?",
        "Which {category} poses are good for {difficulty}s?",
        "I'm a {difficulty}. What {category} poses can I do?"
    ],
    'instructions': [
        "How do I do {pose_name}?",
        "What are the steps for {pose_name}?",
        "Can you explain how to perform {pose_name}?",
        "Walk me through {pose_name}"
    ],
    'contraindications': [
        "I have {condition}. Which poses should I avoid?",
        "What poses are not safe with {condition}?",
        "Which poses should I skip if I have {condition}?"
    ],
    'modifications': [
        "How can I modify {pose_name} if I have {issue}?",
        "What's an easier version of {pose_name}?",
        "I can't do {pose_name} fully. What modifications exist?"
    ],
    'comparison': [
        "What's the difference between {pose1} and {pose2}?",
        "How are {pose1} and {pose2} different?",
        "Should I do {pose1} or {pose2}?"
    ]
}

print(f"Defined {len(question_templates)} question template categories")

Defined 8 question template categories


## Generate Questions Using LLM

We'll feed the LLM pose data chunk-by-chunk and ask it to generate questions grounded in the data.

In [4]:
def generate_questions_for_poses(poses_chunk, num_questions=5):
    """
    Generate questions based on a chunk of pose data.
    
    Args:
        poses_chunk: DataFrame with pose data
        num_questions: Number of questions to generate
    
    Returns:
        List of dicts with question, expected_answer, relevant_pose_ids
    """
    # Convert poses to a readable format
    poses_text = ""
    for _, pose in poses_chunk.iterrows():
        poses_text += f"""\n---\nID: {pose['id']}
Pose: {pose['pose_name']} ({pose['sanskrit_name']})
Category: {pose['category']}
Difficulty: {pose['difficulty_level']}
Benefits: {pose['benefits']}
Contraindications: {pose['contraindications']}
Instructions: {pose['instructions'][:200]}...
Modifications: {pose['modifications'][:150]}...
"""
    
    prompt = f"""You are a yoga expert creating evaluation questions for a RAG system.

Based ONLY on the following yoga pose data, generate {num_questions} diverse, realistic questions that a user might ask.

YOGA POSES:
{poses_text}

For each question, provide:
1. The question text (natural, conversational)
2. A concise expected answer (2-3 sentences max)
3. The relevant pose ID(s) from the data above

Question types to include:
- Identify pose by benefit or characteristic
- Ask about benefits of a specific pose
- Ask about poses in a category or difficulty level
- Ask for instructions on how to do a pose
- Ask about contraindications or safety
- Ask about modifications
- Compare two poses

Return ONLY a JSON array with this structure:
[{{
  "question": "What pose helps with back pain?",
  "expected_answer": "Bridge Pose strengthens back muscles and improves spinal flexibility.",
  "relevant_pose_ids": [9]
}}]

Make questions diverse and natural. Ground ALL answers in the provided data.
"""
    
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a helpful yoga expert creating evaluation data. Always return valid JSON."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.8,
            max_tokens=2000
        )
        
        content = response.choices[0].message.content.strip()
        
        # Try to extract JSON if wrapped in markdown
        if content.startswith('```'):
            content = content.split('```')[1]
            if content.startswith('json'):
                content = content[4:]
        
        questions = json.loads(content)
        return questions
    
    except Exception as e:
        print(f"Error generating questions: {e}")
        return []

print("Question generation function defined")

Question generation function defined


## Generate Questions in Batches

We'll process poses in chunks to generate diverse questions across all poses.

In [5]:
# Generate questions in batches
all_questions = []
chunk_size = 10  # Process 10 poses at a time
questions_per_chunk = 5  # Generate 5 questions per chunk

# Calculate number of chunks needed for ~60 questions
target_questions = 60
num_chunks = min(len(df) // chunk_size, target_questions // questions_per_chunk)

print(f"Generating questions from {num_chunks} chunks of {chunk_size} poses each...")
print(f"Target: ~{num_chunks * questions_per_chunk} questions\n")

for i in range(num_chunks):
    start_idx = i * chunk_size
    end_idx = start_idx + chunk_size
    chunk = df.iloc[start_idx:end_idx]
    
    print(f"Processing chunk {i+1}/{num_chunks} (poses {start_idx+1}-{end_idx})...")
    
    questions = generate_questions_for_poses(chunk, num_questions=questions_per_chunk)
    
    if questions:
        all_questions.extend(questions)
        print(f"  Generated {len(questions)} questions. Total: {len(all_questions)}")
    else:
        print(f"  Failed to generate questions for this chunk")
    
    # Small delay to avoid rate limiting
    import time
    time.sleep(1)

print(f"\nTotal questions generated: {len(all_questions)}")

Generating questions from 12 chunks of 10 poses each...
Target: ~60 questions

Processing chunk 1/12 (poses 1-10)...


  Generated 5 questions. Total: 5


Processing chunk 2/12 (poses 11-20)...


  Generated 5 questions. Total: 10


Processing chunk 3/12 (poses 21-30)...


...

Processing chunk 11/12 (poses 101-110)...


  Generated 5 questions. Total: 70


Processing chunk 12/12 (poses 111-120)...


  Generated 5 questions. Total: 75



Total questions generated: 75


## Review Generated Questions

In [6]:
# Display sample questions
print("Sample questions:\n")
for i, q in enumerate(all_questions[:5], 1):
    print(f"{i}. Q: {q['question']}")
    print(f"   A: {q['expected_answer']}")
    print(f"   Pose IDs: {q['relevant_pose_ids']}")
    print()

Sample questions:

1. Q: What pose is great for improving balance and focus?
   A: Tree Pose and Warrior Pose I are both great for improving balance and focus.
   Pose IDs: [2, 6]

2. Q: What are the benefits of Cobra Pose?
   A: Cobra Pose strengthens the back muscles, opens the chest, and improves flexibility in the shoulders and upper back. It also helps to relieve stress and anxiety.
   Pose IDs: [4]

3. Q: Can you give me some standing poses that are good for beginners?
   A: Mountain Pose, Downward-Facing Dog, and Warrior Pose I are all standing poses that are suitable for beginners.
   Pose IDs: [1, 3, 6]

4. Q: How do I do a Cat-Cow Pose?
   A: Start on your hands and knees, then as you inhale, arch your back and lift your head and tailbone towards the ceiling. As you exhale, round your back and tuck your chin to your chest.
   Pose IDs: [5]

5. Q: What poses should I avoid if I have a recent back injury?
   A: Avoid Cobra Pose, Seated Spinal Twist, and Bridge Pose if you have 

## Build Evaluation Dataset

Format the questions into a structured dataframe for evaluation.

In [7]:
# Create dataframe
ground_truth_df = pd.DataFrame(all_questions)

# Convert relevant_pose_ids to string format for CSV storage
ground_truth_df['relevant_pose_ids'] = ground_truth_df['relevant_pose_ids'].apply(
    lambda x: ','.join(map(str, x)) if isinstance(x, list) else str(x)
)

print(f"Created ground truth dataset with {len(ground_truth_df)} questions")
print(f"\nColumns: {list(ground_truth_df.columns)}")
print(f"\nDataset shape: {ground_truth_df.shape}")

ground_truth_df.head(10)

Created ground truth dataset with 75 questions

Columns: ['question', 'expected_answer', 'relevant_pose_ids']

Dataset shape: (75, 3)


,question,expected_answer,relevant_pose_ids
0,What pose is great for improving balance and f...,Tree Pose and Warrior Pose I are both great fo...,"2,6"
1,What are the benefits of Cobra Pose?,"Cobra Pose strengthens the back muscles, opens...",4
2,Can you give me some standing poses that are g...,"Mountain Pose, Downward-Facing Dog, and Warrio...","1,3,6"
3,How do I do a Cat-Cow Pose?,"Start on your hands and knees, then as you inh...",5
4,What poses should I avoid if I have a recent b...,"Avoid Cobra Pose, Seated Spinal Twist, and Bri...","4,8,9,7"
5,Which beginner-friendly pose is great for open...,"Sphinx Pose strengthens the back muscles, open...",12
6,I have sciatica pain. Are there any poses that...,"Yes, try Pigeon Pose, Seated Forward Fold With...","11,14,15,18"
7,What are the benefits of practicing Warrior Po...,"Warrior Pose II strengthens the legs, hips, an...",17
8,"How do I practice Seated Leg Stretch, also kno...",Sit on the floor with your legs straight out i...,15
9,Are there any contraindications for practicing...,Avoid this pose if you have any neck or should...,16


## Validate Dataset Quality

In [8]:
# Check for missing values
print("Missing values:")
print(ground_truth_df.isnull().sum())
print()

# Check question length distribution
ground_truth_df['question_length'] = ground_truth_df['question'].str.len()
print("Question length statistics:")
print(ground_truth_df['question_length'].describe())
print()

# Check answer length distribution
ground_truth_df['answer_length'] = ground_truth_df['expected_answer'].str.len()
print("Answer length statistics:")
print(ground_truth_df['answer_length'].describe())
print()

# Drop temporary columns
ground_truth_df = ground_truth_df.drop(['question_length', 'answer_length'], axis=1)

# Check pose ID coverage
all_pose_ids = set()
for ids_str in ground_truth_df['relevant_pose_ids']:
    pose_ids = [int(x.strip()) for x in str(ids_str).split(',')]
    all_pose_ids.update(pose_ids)

print(f"Unique poses referenced: {len(all_pose_ids)} out of {len(df)} total poses")
print(f"Coverage: {len(all_pose_ids)/len(df)*100:.1f}%")

Missing values:
question             0
expected_answer      0
relevant_pose_ids    0
dtype: int64

Question length statistics:
count     75.000000
mean      66.546667
std       22.090290
min       25.000000
25%       52.000000
50%       64.000000
75%       76.000000
max      137.000000
Name: question_length, dtype: float64

Answer length statistics:
count     75.000000
mean     152.480000
std       48.501145
min       65.000000
25%      122.500000
50%      144.000000
75%      181.000000
max      323.000000
Name: answer_length, dtype: float64

Unique poses referenced: 89 out of 202 total poses
Coverage: 44.1%


## Save Ground Truth Dataset

In [9]:
# Save to CSV
output_path = '../data/ground_truth.csv'
ground_truth_df.to_csv(output_path, index=False)

print(f"✓ Saved ground truth dataset to {output_path}")
print(f"✓ Total questions: {len(ground_truth_df)}")
print(f"✓ Ready for retrieval and RAG evaluation!")

✓ Saved ground truth dataset to ../data/ground_truth.csv
✓ Total questions: 75
✓ Ready for retrieval and RAG evaluation!


## Summary

This notebook generated a synthetic evaluation dataset with:
- Diverse question types (identify pose, benefits, category, difficulty, instructions, contraindications, modifications, comparisons)
- Questions grounded in actual pose data
- Expected answers for each question
- Relevant pose IDs for verification

The dataset can now be used to:
1. Evaluate retrieval approaches (hit rate, MRR)
2. Evaluate RAG generation quality
3. Compare different models and approaches